# 🤖 Huấn Luyện Học Tăng Cường (RL) Cho Robot Tự Cân Bằng 2 Bánh

**Quy chuẩn 6 bước chuyên nghiệp:**
1. **Mô phỏng mô hình vật lý (Simulation Environment):** Custom Gymnasium Env với động lực học phi tuyến, trễ phần cứng và vùng chết động cơ (Ts = 5 ms / 200 Hz).
2. **Không gian Trạng thái (Observation Space):** Vector 4D $[\theta, \dot{\theta}, x, v]^T$.
3. **Không gian Hành động (Action Space):** Hành động liên tục $a_t \in [-1.0, 1.0]$ quy đổi ra PWM $[-999, +999]$.
4. **Hàm thưởng chống rung giật (Reward Shaping):** Thưởng tồn tại $+1$, phạt nghiêng $-\theta^2$, phạt giật động cơ $-(a_t - a_{t-1})^2$, phạt trôi vị trí $-x^2$.
5. **Huấn luyện với Stable-Baselines3 (PPO):** Mạng nơ-ron 2 tầng ẩn `net_arch=[64, 64]` tối ưu thời gian thực.
6. **Xuất mô hình C Header (Deployment Prep):** Xuất trực tiếp sang file C Header `rl_sb3_stm32.h` sẵn sàng nạp lên STM32L476RG.

--- 
### Bước 0: Cài đặt các thư viện RL chuẩn (Gymnasium, Stable-Baselines3, PyTorch)

In [ ]:
!pip install gymnasium stable-baselines3 torch matplotlib pyyaml tensorboard

--- 
### Bước 1 → 4: Khởi tạo Môi trường Gymnasium & Kiểm thử Hàm thưởng

Kiểm tra quan sát 4D $[\theta, \dot{\theta}, x, v]$, không gian hành động và hàm thưởng phạt rung giật.

In [ ]:
import yaml
from src.environment import SelfBalancingGymEnv

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

env = SelfBalancingGymEnv(config)
obs, info = env.reset(seed=42)
print("✅ Không gian quan sát (Observation Space):", env.observation_space)
print("✅ Không gian hành động (Action Space):     ", env.action_space)
print("✅ Trạng thái ban đầu (theta, dtheta, x, v): ", obs)

--- 
### Bước 5: Huấn luyện Thuật toán PPO với Stable-Baselines3

Khởi tạo mạng `net_arch=[64, 64]` và huấn luyện 200,000 timesteps.

In [ ]:
!python train.py --config configs/default.yaml --timesteps 200000

--- 
### Bước 6: Chạy Mô phỏng Suy luận & Hiển thị Đồ thị Phản hồi cân bằng

In [ ]:
!python inference.py

from IPython.display import Image, display
display(Image(filename='outputs/figures/rl_inference_test.png'))

--- 
### Bước 7: Tải file C Header `rl_sb3_stm32.h` về nhúng vào STM32CubeIDE

File header này không phụ thuộc bất kỳ thư viện ngoài nào (Zero-Dependency), tính toán Forward Pass chỉ tốn ~30-40 $\mu\text{s}$ trên ARM Cortex-M4 (80 MHz).

In [ ]:
try:
    from google.colab import files
    files.download('outputs/firmware_headers/rl_sb3_stm32.h')
    print("⬇️ Đang tải file C Header về máy tính...")
except ImportError:
    print("File đã được xuất sẵn tại: outputs/firmware_headers/rl_sb3_stm32.h")